# Super-resolution figures -- SyMTRS

Four models (autoencoder, SRCNN, SRGAN, SwinIR) at three scales (x2, x4, x8). Twelve runs;
eleven scored. The figures are written to be *chosen from*: each one is a different answer
to "what does super-resolution buy on this dataset", and the paper needs two or three of
them, not all twelve.

| Figure | Question it answers | Shape |
| --- | --- | --- |
| 1 | How much does each model recover, and how fast does it decay with scale? | line, 4 series, one panel |
| 2 | What is the input, and what does each model return, at x2/x4/x8? | 1 scene x 3 scales x 6 columns |
| 3 | Does it hold up on different ground (vegetation / asphalt / concrete, x4)? | 3 scenes x 6 columns |
| 4 | Where is the error, and is it missing detail or invented detail? | crops + error maps |
| 5 | The table as numbers: dots (5a, accurate) or bars (5b, conventional) | 3 panels |
| 6 | How wide is the spread behind each mean? | ridge densities |
| 7 | Do the models fail on the same images? | CDF per scale |
| 8 | How often is a model worse than plain bicubic? | gain histograms |
| 9 | Which model learns fastest / longest? | validation curves |
| 10 | Which model wins image by image? | win-rate matrices |
| 11 | What does the quality cost in parameters? | scatter |
| 12 | The results table as a heat map | 3 annotated matrices |
| 13 | Where the detail comes from: power spectra | 3 panels, log curves |
| 14 | Output vs ground truth, split down the middle (project page) | 4 panes |
| 15 | The same detail at x2/x4/x8 on vegetation, asphalt and concrete (best model per scale) | 3 scenes x 3 scales |
| 16 | All four models at one scale, on the same three windows | 3 files, 3 scenes x 4 models |
| 17 | **The paper's training figure**: the loss each run selected, and the curves it came from | 2 panels, one row |

Nothing is re-scored here. The aggregates are the runs' own `summary.json` (written by
`symtrs.model.superres` from the trainer's `test_metrics.json`); the per-image rows come
from `notebooks/sr_per_image.py`, which walks the test split through the trainer's own
`super_resolve`; the crops come from `notebooks/sr_render.py`. Both scripts are cheap to
re-run and neither writes a result -- if a per-image number disagrees with the aggregate,
the aggregate is right and the script is wrong.

**What the low-resolution images are, and what "bicubic" means here.** The `_lr` folders
are not resampled 2048 px frames: they come from the simulated camera at the lower
resolution, with that camera's own sampling and sensor noise. So everywhere below
"bicubic" is the bicubic *upsampling of the real LR image* to the ground-truth size -- the
baseline a model is measured against, never the operator that made its input. This matters
when reading the gain-over-bicubic numbers: against the real LR, bicubic agrees with a
resample of the ground truth only to ~24-28 dB (residual sigma 40-80/255, and at x2 the
camera image is locally *sharper* than the resample), so the floor is not the analytic
downsample bicubic is tuned to invert. The gains are honest for this dataset; they are not
directly comparable with SR papers that degrade their HR images with `imresize`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the "3d" projection)
from matplotlib import colors as mcolors
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from PIL import Image


def _repo_root() -> Path:
    """The repository, whether this runs as a notebook (no `__file__`) or as a script."""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()
    for anchor in (start, *start.parents):
        if (anchor / "symtrs" / "model").is_dir():
            return anchor
    raise SystemExit(f"cannot find the SyMTRS repository above {start}")


REPO = _repo_root()
RUNS = REPO / "runs" / "campaign2"
OUT = REPO / "notebooks" / "superresolution_figures"
OUT.mkdir(exist_ok=True)
QUAL = Path("/tmp/srqual")  # written by notebooks/sr_render.py

# ---------------------------------------------------------------------------
# Palette: the segmentation and shadow notebooks' slots, kept in the same order so a model
# has one colour across the paper (blue / orange / green / amber). Scale is an ordered
# quantity and is *faceted*, never coloured: a second colour scale on the same panel is how
# a figure stops being readable.
# ---------------------------------------------------------------------------
SURFACE = "#ffffff"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
HAIR, AXIS = "#e1e0d9", "#c3c2b7"
BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
             "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ = mcolors.LinearSegmentedColormap.from_list("symtrs_blue", BLUE_RAMP)
SEQ_ERR = mcolors.LinearSegmentedColormap.from_list("symtrs_err", ["#ffffff"] + BLUE_RAMP)
#: win rate: above 0.5 the row model wins, below it loses. Two hues + a neutral midpoint,
#: because the value has a meaningful centre; not a rainbow, which would put a hue on 50 %.
DIV = mcolors.LinearSegmentedColormap.from_list(
    "symtrs_win", ["#b8461f", "#eb6834", "#f6f5f2", "#5598e7", "#1c5cab"])

MODELS = [("autoencoder", "Autoencoder", "#2a78d6"),
          ("srcnn", "SRCNN", "#eb6834"),
          ("srgan", "SRGAN", "#1baf7a"),
          ("swinir", "SwinIR", "#eda100")]
SCALES = (2, 4, 8)
COLOR = {m: c for m, _, c in MODELS}
LABEL = {m: l for m, l, _ in MODELS}
NAMES = [m for m, _, _ in MODELS]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": HAIR, "grid.linewidth": 0.7,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2.0, "lines.markersize": 6.5,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def tidy(ax, grid_axis="y"):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis=grid_axis)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)


def strip(ax, image, **kw):
    """An image pane with no ticks, no grid and no frame."""
    kw.setdefault("interpolation", "lanczos")  # the LR column overrides this with "nearest"
    ax.imshow(image, **kw)
    ax.set_xticks([]), ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def badge(ax, text, loc="lower left", size=7.2):
    x, ha = (0.035, "left") if "left" in loc else (0.965, "right")
    y, va = (0.04, "bottom") if "lower" in loc else (0.96, "top")
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=size, color=INK,
            bbox=dict(boxstyle="round,pad=0.22", fc=(1, 1, 1, 0.85), ec="none"))


def save(fig, name):
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight", pad_inches=0.01)
    fig.savefig(OUT / f"{name}.png", dpi=200, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)
    print(f"wrote superresolution_figures/{name}.pdf / .png")
    try:  # inline when this runs as a notebook; no-op headless
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(OUT / f"{name}.png"), width=760))
    except Exception:
        pass

In [ ]:
# ---------------------------------------------------------------------------
# The runs: aggregate test rows, per-image rows, validation curves
# ---------------------------------------------------------------------------
def run_name(model, scale):
    return f"{model}_x{scale}"


def load_json(path):
    return json.loads(Path(path).read_text()) if Path(path).is_file() else None


TEST, PER, CURVE = {}, {}, {}
for model in NAMES:
    for scale in SCALES:
        d = RUNS / run_name(model, scale)
        summary = load_json(d / "summary.json")
        if summary and summary.get("test"):
            TEST[(model, scale)] = summary["test"]
        # Only a finished, scored run's per-image rows are used: a run still training has a
        # best.pt that moves under the scoring script, so its rows would not be reproducible.
        rows = load_json(d / "per_image_test.json") if (model, scale) in TEST else None
        if rows:
            PER[(model, scale)] = {Path(r["image"]).name: r for r in rows}
        csv = d / "metrics.csv"
        if csv.is_file():
            lines = csv.read_text().splitlines()
            head = lines[0].split(",")
            CURVE[(model, scale)] = [
                {k: float(v) if v not in ("", "nan") else float("nan") for k, v in zip(head, ln.split(","))}
                for ln in lines[1:] if ln.strip()
            ]
SCORED = [k for k in TEST]
PER_KEYS = {}
for _s in SCALES:
    _have = [set(PER[(_m, _s)]) for _m in NAMES if (_m, _s) in PER]
    PER_KEYS[_s] = sorted(set.intersection(*_have)) if _have else []

print(f"test rows: {len(SCORED)}/12 runs")
print(f"\n{'':16}{'params':>8}{'PSNR':>8}{'SSIM':>8}{'MSE':>10}   images")
for model in NAMES:
    for scale in SCALES:
        t = TEST.get((model, scale))
        if not t:
            continue
        print(f"{LABEL[model] + ' x' + str(scale):<16}" + " " * 0 +
              f"{t['psnr']:8.2f}{t['ssim']:8.3f}{t['mse']:10.5f}{t['images']:>9}")
print("\nper-image rows (for the distributions):")
for scale in SCALES:
    have = [LABEL[m] for m in NAMES if (m, scale) in PER]
    print(f"  x{scale}: {len(PER_KEYS[scale])} images, {', '.join(have) or 'none yet'}")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 1 -- quality against scale
# ---------------------------------------------------------------------------
# The one-panel version of the results table: PSNR on the left axis, SSIM as a second
# panel rather than a second y-axis (two scales, one panel, is the mistake this figure
# exists to avoid). Direct labels at the right end, so the reader does not have to look
# up a legend colour.

In [ ]:
def fig_scale_lines():
    metrics = [("psnr", "PSNR (dB)"), ("ssim", "SSIM")]
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.5))
    for ax, (key, ylabel) in zip(axes, metrics):
        ends = []
        for model in NAMES:
            pts = [(s, TEST[(model, s)][key]) for s in SCALES if (model, s) in TEST]
            if not pts:
                continue
            xs, ys = zip(*pts)
            ax.plot(xs, ys, marker="o", color=COLOR[model], label=LABEL[model],
                    markeredgecolor="white", markeredgewidth=1.2, zorder=3)
            ends.append([ys[-1], float(ys[-1]), LABEL[model], COLOR[model]])
        # Direct labels instead of a legend, pushed apart by a minimum gap: four models at
        # x8 sit within 0.4 dB of each other and would otherwise print on top of one another.
        # The label keeps its model's colour and a hairline back to the marker it belongs to,
        # so a label that had to move is still unambiguous.
        ends.sort(key=lambda e: e[0])
        gap = 0.42 if key == "psnr" else 0.014
        for i in range(1, len(ends)):
            if ends[i][1] - ends[i - 1][1] < gap:
                ends[i][1] = ends[i - 1][1] + gap
        for y, y_text, text, color in ends:
            ax.plot([8.15, 8.95], [y, y_text], color=color, lw=0.5, alpha=0.55, zorder=2)
            ax.text(9.05, y_text, text, va="center", fontsize=8.5, color=INK2)
        ax.set_xticks(SCALES, [f"x{s}" for s in SCALES])
        ax.set_xlabel("upscaling factor")
        ax.set_title(ylabel.split(" (")[0], loc="left")
        ax.set_ylabel(ylabel)
        ax.set_xlim(1.6, 11.4)
        tidy(ax)
    save(fig, "sr_fig1_scale_lines")


fig_scale_lines()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 2 -- one window, three scales
# ---------------------------------------------------------------------------
# Three rows, one per scale, six columns: the input, the four models, the ground truth. One
# scene carried down the ladder, because the question is not "which scene" but "what happens
# to the same place when the camera has a quarter of the pixels". Every pane shows the same
# physical window of the 2048 px frame, so the columns are comparable and the rows read as
# the loss of information.
#
# The first column is the input at its own pixel size, upscaled nearest-neighbour for
# display: 256 px at x2, 128 at x4, 64 at x8. Nearest, not bicubic, so a reader sees the grid
# the network was handed rather than a resample that flatters it; it carries the pixel size
# instead of a PSNR, because a blocky upsample is not scored. The Autoencoder and SRCNN are
# handed a bicubic upsample of this patch (that upsample is what they refine); SRGAN and
# SwinIR take it as it stands. Badges on the model columns are the PSNR of *that crop* (RGB,
# the written image), not of the whole frame.

In [ ]:
MANIFEST = load_json(QUAL / "manifest.json")
TILES = []
if MANIFEST:
    pool = [k for k in MANIFEST["keys"]
            if all(f"x{s}" in MANIFEST["boxes"][k] for s in SCALES)]
    TILES = [pool[i] for i in np.linspace(0, len(pool) - 1, min(4, len(pool))).round().astype(int)]
    print(f"{len(pool)} rendered scenes in {QUAL}; showing {len(TILES)}")
else:
    print(f"no crops in {QUAL}: run  python notebooks/sr_render.py --device cuda:1")


def crop(model, key, scale):
    return np.asarray(Image.open(QUAL / f"x{scale}" / f"{run_name(model, scale)}_{key}.png").convert("RGB"))


def truth(key, scale):
    return np.asarray(Image.open(QUAL / f"x{scale}" / f"hr_{key}.png").convert("RGB"))


def bicubic(key, scale):
    return np.asarray(Image.open(QUAL / f"x{scale}" / f"bicubic_{key}.png").convert("RGB"))


def has_crop(model, key, scale):
    """False where a run has no crop at that scale -- SwinIR has no x2 checkpoint yet, and a
    column of whitespace beats a KeyError halfway through a figure."""
    return (QUAL / f"x{scale}" / f"{run_name(model, scale)}_{key}.png").is_file()

In [ ]:
def lr_input(key, scale, size=512):
    img = Image.open(QUAL / f"x{scale}" / f"lr_{key}")
    if img.size != (size, size):
        img = img.resize((size, size), Image.NEAREST)
    return np.asarray(img.convert("RGB"))


def has_lr(key, scale):
    """False where notebooks/sr_lr_crops.py has not cut the input patch for a scene."""
    return (QUAL / f"x{scale}" / f"lr_{key}").is_file()


CONTENT = (MANIFEST or {}).get("content", {})


def _content_features(key):
    """Share of green, dark and bright low-saturation pixels -- a cheap scene classifier.

    Measured on the whole 1024 px frame, not on the crop: the figure shows 1/16 of the scene,
    so a parking lot whose most detailed window happens to hold a tree would classify as
    vegetation. `notebooks/sr_lr_crops.py` writes the signature into the manifest; the crop
    fallback is for a manifest written by an older `sr_render.py`.
    """
    if key in CONTENT:
        return CONTENT[key]
    img = truth(key, 8).astype(np.float32) / 255.0
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    value, sat = img.mean(-1), img.max(-1) - img.min(-1)
    return {
        "vegetation": float(((g > r + 0.03) & (g > b + 0.03) & (value > 0.12)).mean()),
        "asphalt": float(((value < 0.38) & (sat < 0.14)).mean()),
        "concrete": float(((value > 0.52) & (sat < 0.12)).mean()),
    }


# The three content rows are chosen by hand from the rendered pool, and the reason is worth
# keeping: a colour-share classifier over the whole frame ranked a parking lot first for
# "asphalt" whose most detailed window turned out to be a tree canopy -- it read "asphalt",
# the page showed foliage. A detail-driven window and a content-driven choice are two
# different decisions, so the last word has to be the pixels. `_content_features` is kept as
# the audit: it prints what each chosen window is made of, so a scene that stops matching its
# label after a re-render says so instead of quietly mislabelling a row.
CONTENT_ROWS = {
    "foliage": "rgb_day_2048_loc490_rot179deg_h01_x+061722_y-052400_z+010046.png",
    "road": "rgb_day_2048_loc465_rot270deg_h01_x+054441_y-007217_z+011000.png",
    "roof": "rgb_day_2048_loc484_rot172deg_h03_x+056599_y+010856_z+016068.png",
}


def _crop_features(key):
    """The same three shares, on the window the figure will actually show."""
    img = truth(key, 8).astype(np.float32) / 255.0
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    value, sat = img.mean(-1), img.max(-1) - img.min(-1)
    return {
        "vegetation": float(((g > r + 0.03) & (g > b + 0.03) & (value > 0.12)).mean()),
        "asphalt": float(((value < 0.38) & (sat < 0.14)).mean()),
        "concrete": float(((value > 0.52) & (sat < 0.12)).mean()),
    }


def content_scenes(audit=False):
    """The three content rows (figures 2, 3, 15, 16): label -> scene key."""
    rows = {label: key for label, key in CONTENT_ROWS.items()
            if key in MANIFEST["boxes"] and all(f"x{s}" in MANIFEST["boxes"][key] for s in SCALES)}
    if audit:
        for label, key in rows.items():
            scene, crop = _content_features(key), _crop_features(key)
            print(f"  {label:7s} {key[12:34]}  crop " +
                  "  ".join(f"{c}={crop[c]:.2f}" for c in crop))
    return rows


GRID_COLS = [("lr", "LR input")] + [(m, LABEL[m]) for m in NAMES] + [("hr", "Ground truth")]


def _grid(rows):
    """A qualitative grid: same 6-column layout whatever the rows are (scales or contents)."""
    width = 7.2
    cell = width / len(GRID_COLS)
    return plt.subplots(len(rows), len(GRID_COLS),
                        figsize=(width, cell * len(rows) + 0.22), squeeze=False)


def _grid_pane(ax, source, key, scale, psnr):
    """One pane: the input, a model output, or the ground truth. Blanks a missing run."""
    if source == "lr":
        if not has_lr(key, scale):
            ax.axis("off")
            return
        strip(ax, lr_input(key, scale), interpolation="nearest")
        # The patch's own size, not the frame's: it is what the blockiness in the pane is.
        badge(ax, f"{512 // scale} px")
    elif source == "hr":
        strip(ax, truth(key, scale))
    elif not has_crop(source, key, scale):
        ax.axis("off")
    else:
        strip(ax, crop(source, key, scale))
        badge(ax, f"{psnr[run_name(source, scale)]:.1f} dB")


def _grid_finish(fig, name, left=0.002):
    """`left` is the gutter a row label needs: "x2" fits in none, "foliage" does not."""
    fig.subplots_adjust(left=left, right=0.998, top=1 - 0.22 / fig.get_figheight(),
                        bottom=0.002, hspace=0.02, wspace=0.02)
    save(fig, name)


def fig_scales(key=None):
    key = key or TILES[0]
    fig, axes = _grid(SCALES)
    for i, scale in enumerate(SCALES):
        psnr = MANIFEST["boxes"][key][f"x{scale}"]["psnr"]
        for j, (source, title) in enumerate(GRID_COLS):
            _grid_pane(axes[i, j], source, key, scale, psnr)
            if i == 0:
                axes[i, j].set_title(title, fontsize=9, loc="center")
        axes[i, 0].set_ylabel(f"x{scale}", color=INK, fontsize=11, labelpad=4)
    _grid_finish(fig, "sr_fig2_grid")


if TILES:
    fig_scales()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 3 -- does it hold up on different ground?
# ---------------------------------------------------------------------------
# The same six columns at x4, one row per kind of ground: foliage, a road with vehicles, a
# roof with skylights (see `CONTENT_ROWS` above for how they are chosen and audited). The
# scales figure shows what resolution costs; this one shows that the ranking between models is
# about content -- SwinIR's margin is on edges that exist (kerbs, markings, skylights) and
# vanishes on texture no model recovers.

In [ ]:
def fig_content(scale=4):
    rows = list(content_scenes(audit=True).items())
    if not rows:
        print("no rendered scenes: run  python notebooks/sr_render.py --device cuda:1")
        return
    fig, axes = _grid(rows)
    for i, (cls, key) in enumerate(rows):
        psnr = MANIFEST["boxes"][key][f"x{scale}"]["psnr"]
        for j, (source, title) in enumerate(GRID_COLS):
            _grid_pane(axes[i, j], source, key, scale, psnr)
            if i == 0:
                axes[i, j].set_title(title, fontsize=9, loc="center")
        axes[i, 0].set_ylabel(cls, color=INK, fontsize=10, labelpad=4)
    _grid_finish(fig, "sr_fig3_grid_x4", left=0.055)


if TILES:
    fig_content(4)

In [ ]:
# ---------------------------------------------------------------------------
# Figure 4 -- where the error is: crops and error maps at x8
# ---------------------------------------------------------------------------
# The scene where SwinIR and SRGAN disagree most at x8 (their gap in crop PSNR is largest),
# so the two failure modes sit side by side: SRGAN's over-sharp, invented edges against
# SwinIR's smooth, sometimes misplaced texture. Top row the pixels, bottom row |output -
# target| in display range, one colour scale for every panel; the first map is bicubic,
# i.e. what the scale itself costs before any network.

In [ ]:
def fig_errors():
    scale = 8
    keys = [k for k in MANIFEST["keys"] if f"x{scale}" in MANIFEST["boxes"][k]]
    if not keys:
        print(f"no x{scale} crops to compare")
        return
    key = max(keys, key=lambda k: abs(MANIFEST["boxes"][k][f"x{scale}"]["psnr"]["swinir_x8"]
                                      - MANIFEST["boxes"][k][f"x{scale}"]["psnr"]["srgan_x8"]))
    target = truth(key, scale).astype(np.float32) / 255.0
    sources = [("bicubic", "Bicubic baseline")] + [(m, LABEL[m]) for m in NAMES]
    errs = {}
    for source, _ in sources:
        img = bicubic(key, scale) if source == "bicubic" else crop(source, key, scale)
        errs[source] = np.abs(img.astype(np.float32) / 255.0 - target).mean(-1)
    vmax = float(np.percentile(errs["bicubic"], 99))

    fig, axes = plt.subplots(2, len(sources), figsize=(7.2, 7.2 * 0.45), squeeze=False)
    for j, (source, title) in enumerate(sources):
        img = bicubic(key, scale) if source == "bicubic" else crop(source, key, scale)
        strip(axes[0, j], img)
        axes[0, j].set_title(title, fontsize=9, loc="center")
        badge(axes[0, j], f"{MANIFEST['boxes'][key][f'x{scale}']['psnr'][source if source == 'bicubic' else run_name(source, scale)]:.1f} dB")
        strip(axes[1, j], errs[source], cmap=SEQ_ERR, vmin=0, vmax=vmax)
        badge(axes[1, j], f"MAE {errs[source].mean():.3f}")
    fig.subplots_adjust(left=0.002, right=0.93, top=0.94, bottom=0.02, hspace=0.03, wspace=0.03)
    cax = fig.add_axes((0.945, 0.06, 0.012, 0.38))
    bar = fig.colorbar(plt.cm.ScalarMappable(cmap=SEQ_ERR, norm=mcolors.Normalize(0, vmax)), cax=cax)
    bar.outline.set_visible(False)
    bar.ax.tick_params(labelsize=7.5, length=2)
    bar.set_label("|output - target| (RGB)", fontsize=8, color=INK2)
    save(fig, "sr_fig4_errors_x8")


if MANIFEST:
    fig_errors()

## Figure 5 -- the table as numbers

The same three metrics twice, because they want different forms. **5a (dots)** is the one
to use: PSNR and SSIM live in a 6 dB / 0.25 band, and a bar chart's mandatory zero baseline
throws most of that away. The thin grey segment joins the four models at a scale (how much
the choice of model matters); the whisker is the standard error of the mean over the sampled
images, and the per-image spread it comes from is figure 6. **5b
(bars)** is the conventional form, zero-based, for when a bar chart is expected. Neither
prints the numbers on the marks -- four values 0.1 dB apart turn into a smear of rotated
digits; figure 12 is the version that carries the digits.

In [ ]:
def _metric_values(key):
    """(scale, value, standard error of the mean) per model, only for runs with a test row."""
    out = {}
    for model in NAMES:
        pts = []
        for s in SCALES:
            if (model, s) not in TEST:
                continue
            # The whisker is the standard error of the mean, not the per-image spread: on 300
            # images a +-1 sigma bar is 3 dB wide and swallows every difference in the panel.
            # The spread itself is figure 6, where it belongs.
            err = 0.0
            if (model, s) in PER:
                rows = [r[key] for r in PER[(model, s)].values()]
                err = float(np.std(rows) / np.sqrt(len(rows)))
            pts.append((s, TEST[(model, s)][key], err))
        if pts:
            out[model] = pts
    return out


def _metrics_layout(kind):
    metrics = [("psnr", "PSNR (dB)", "{:.2f}"), ("ssim", "SSIM", "{:.3f}"), ("mse", "MSE", "{:.4f}")]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.7))
    width = 0.2
    for ax, (key, label, fmt) in zip(axes, metrics):
        data = _metric_values(key)
        if kind == "dots":
            # the spread within one scale: one light segment from the lowest to the highest
            for i, s in enumerate(SCALES):
                vals = [v for model in data for (sc, v, _) in data[model] if sc == s]
                if vals:
                    ax.hlines(min(vals), i - 0.28, i + 0.28, color=HAIR, lw=1.4, zorder=1)
        for k, (model, pts) in enumerate(data.items()):
            pos = np.array([SCALES.index(s) + (k - 1.5) * width for s, _, _ in pts])
            vals = np.array([v for _, v, _ in pts])
            err = np.array([e for _, _, e in pts])
            if kind == "bars":
                ax.bar(pos, vals, width * 0.9, color=COLOR[model], label=LABEL[model], zorder=2)
            else:
                ax.plot(pos, vals, "o", ms=5.5, color=COLOR[model], mec="white", mew=1.1, zorder=3)
            ax.errorbar(pos, vals, yerr=err, fmt="none", ecolor=MUTED, lw=0.7, alpha=0.55,
                        capsize=0, zorder=2)
        ax.set_xticks(range(len(SCALES)), [f"x{s}" for s in SCALES])
        ax.set_title(label.split(" (")[0], loc="left")
        tidy(ax)
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + 0.12 * (hi - lo))  # headroom for the whiskers, not the labels
    axes[0].set_ylabel("value")
    handles = [Line2D([], [], color=COLOR[m], marker="o" if kind == "dots" else None,
                      lw=0 if kind == "dots" else 6, label=LABEL[m]) for m in NAMES]
    fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0),
               handlelength=1.2, columnspacing=1.2)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    save(fig, f"sr_fig5{kind}")


_metrics_layout("dots")
_metrics_layout("bars")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 6 -- the spread behind each mean
# ---------------------------------------------------------------------------
# Per-image PSNR, one ridge per model, one panel per scale. The means are the bars in
# figure 5; these are the distributions they came from. Two models whose means are 0.3 dB
# apart overlap almost completely -- that is what a win-rate matrix (figure 10) is for.

In [ ]:
def density(values, grid, sigma=0.22):
    """Gaussian-smoothed density on `grid` (dB), no scipy."""
    d = grid[1] - grid[0]
    half = int(4 * sigma / d) + 1
    ker = np.exp(-0.5 * ((np.arange(-half, half + 1) * d) / sigma) ** 2)
    ker /= ker.sum()
    h, _ = np.histogram(values, bins=np.r_[grid[0] - d / 2, grid + d / 2])
    return np.convolve(h / (len(values) * d), ker, mode="same")


def fig_ridge():
    fig, axes = plt.subplots(1, len(SCALES), figsize=(7.2, 2.9), sharey=True)
    for ax, scale in zip(axes, SCALES):
        keys = PER_KEYS[scale]
        if not keys:
            ax.set_visible(False)
            continue
        values = {m: np.array([PER[(m, scale)][k]["psnr"] for k in keys])
                  for m in NAMES if (m, scale) in PER}
        grid = np.linspace(min(min(v) for v in values.values()) - 1,
                           max(max(v) for v in values.values()) + 1, 300)
        for i, model in enumerate([m for m in NAMES if m in values]):
            d = density(values[model], grid)
            d = d / d.max() * 0.85
            base = i
            ax.fill_between(grid, base, base + d, color=COLOR[model], alpha=0.22, lw=0)
            ax.plot(grid, base + d, color=COLOR[model], lw=1.4)
            ax.axhline(base, color=HAIR, lw=0.7, zorder=0)
            ax.plot([np.median(values[model])] * 2, [base, base + np.interp(np.median(values[model]), grid, d)],
                    color=COLOR[model], lw=0.9, ls=(0, (2, 2)))
        shown = [m for m in NAMES if m in values]
        ax.set_yticks(range(len(shown)), [LABEL[m] for m in shown])
        ax.set_ylim(-0.2, len(values) - 0.05)
        ax.set_xlabel("PSNR per image (dB)")
        ax.set_title(f"x{scale}", loc="left")
        tidy(ax, grid_axis="x")
    fig.tight_layout()
    save(fig, "sr_fig6_ridge")


fig_ridge()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 7 -- how bad is the bad tail
# ---------------------------------------------------------------------------
# The cumulative distribution of the same per-image PSNRs. A model that is better on
# average but has a long left tail is worse for a mosaic: the worst 5 % of tiles is what a
# reader notices. The dashed line marks that tail.

In [ ]:
def fig_cdf():
    fig, axes = plt.subplots(1, len(SCALES), figsize=(7.2, 2.5), sharey=True)
    for ax, scale in zip(axes, SCALES):
        keys = PER_KEYS[scale]
        if not keys:
            ax.set_visible(False)
            continue
        for model in NAMES:
            if (model, scale) not in PER:
                continue
            v = np.sort([PER[(model, scale)][k]["psnr"] for k in keys])
            ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post",
                    color=COLOR[model], lw=1.8, label=LABEL[model])
        ax.axhline(0.05, color=MUTED, lw=0.8, ls=(0, (3, 2)))
        ax.text(ax.get_xlim()[0] + 0.3, 0.075, "worst 5 %", fontsize=7, color=MUTED)
        ax.set_xlabel("PSNR per image (dB)")
        ax.set_title(f"x{scale}", loc="left")
        ax.set_ylim(0, 1)
        tidy(ax)
    axes[0].set_ylabel("share of test images")
    axes[0].legend(loc="lower right", handlelength=1.2)
    fig.tight_layout()
    save(fig, "sr_fig7_cdf")


fig_cdf()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 8 -- is a network worth it over bicubic at all?
# ---------------------------------------------------------------------------
# Per-image gain over the bicubic baseline that the network itself is built on, for the
# same crops the metrics used. Everything left of zero is an image the network made worse
# than a resampling filter; at x8 that is not a curiosity, it is a large share of the
# split, and the figure is the honest counterweight to the means in figure 5.

In [ ]:
def fig_gain():
    fig, axes = plt.subplots(1, len(SCALES), figsize=(7.2, 2.9), sharey=True)
    for ax, scale in zip(axes, SCALES):
        keys = PER_KEYS[scale]
        if not keys:
            ax.set_visible(False)
            continue
        gains = {m: np.array([PER[(m, scale)][k]["psnr"] - PER[(m, scale)][k]["bicubic"]["psnr"]
                              for k in keys]) for m in NAMES if (m, scale) in PER}
        lo = min(g.min() for g in gains.values()) - 0.5
        hi = max(g.max() for g in gains.values()) + 0.5
        bins = np.linspace(lo, hi, 60)
        for model, g in gains.items():
            ax.hist(g, bins=bins, histtype="step", color=COLOR[model], lw=1.5,
                    label=f"{LABEL[model]} ({100 * (g > 0).mean():.0f} % better)")
        ax.axvline(0, color=INK, lw=0.9)
        ax.set_xlabel("PSNR gain over bicubic (dB)")
        ax.set_title(f"x{scale}", loc="left")
        tidy(ax)
    axes[0].set_ylabel("images")
    axes[0].legend(loc="upper left", handlelength=1.2, fontsize=8)
    fig.tight_layout()
    save(fig, "sr_fig8_gain")


fig_gain()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 9 -- validation curves
# ---------------------------------------------------------------------------
# Validation PSNR against epoch, one panel per scale, so the comparison is always within
# a scale (the scales are not comparable on one axis). The dot marks the best epoch, which
# is the checkpoint every test number above was scored from. SRGAN is noisy by design: its
# loss is adversarial, and its best epoch is the one the test row used.

In [ ]:
def _smooth(y, w=3):
    """Edge-padded moving average: the epoch-to-epoch wobble of an adversarial loss hides its
    trend, which is the only thing the plot is for."""
    y = np.asarray(y, dtype=float)
    y = np.where(np.isfinite(y), y, np.nanmean(y))
    return np.convolve(np.pad(y, w // 2, mode="edge"), np.ones(w) / w, mode="valid")


def fig_curves():
    fig, axes = plt.subplots(1, len(SCALES), figsize=(7.2, 2.5), sharey=False)
    for ax, scale in zip(axes, SCALES):
        for model in NAMES:
            rows = CURVE.get((model, scale))
            if not rows:
                continue
            ep = np.array([r["epoch"] for r in rows])
            val = np.array([r["val_psnr"] for r in rows])
            ax.plot(ep, val, color=COLOR[model], lw=0.7, alpha=0.32, zorder=2)
            ax.plot(ep, _smooth(val), color=COLOR[model], lw=1.9, zorder=3)
            i = int(np.nanargmax(val))
            ax.plot(ep[i], val[i], "o", ms=5, color=COLOR[model], mec="white", mew=1.1, zorder=4)
        ax.set_xlabel("epoch")
        ax.set_title(f"x{scale}", loc="left")
        tidy(ax)
    axes[0].set_ylabel("validation PSNR (dB)")
    handles = [Line2D([], [], color=COLOR[m], lw=2, label=LABEL[m]) for m in NAMES]
    fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0),
               handlelength=1.2, columnspacing=1.2)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    save(fig, "sr_fig9_curves")


fig_curves()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 10 -- who wins, image by image
# ---------------------------------------------------------------------------
# At each scale, the share of test images on which the row model has the higher PSNR than
# the column model. 50 % is a tie (shown as the neutral centre); a row that is dark blue
# throughout is the better model on nearly every image, and two models that merely trade
# places sit near 50 % everywhere -- which the means cannot tell you.

In [ ]:
def fig_winmatrix():
    scales = [s for s in SCALES if len(PER_KEYS[s])]
    if not scales:
        print("no per-image rows yet")
        return
    fig, axes = plt.subplots(1, len(scales), figsize=(7.2, 2.35), squeeze=False)
    for ax, scale in zip(axes[0], scales):
        keys = PER_KEYS[scale]
        models = [m for m in NAMES if (m, scale) in PER]
        a = np.array([[PER[(m, scale)][k]["psnr"] for k in keys] for m in models])
        win = np.zeros((len(models), len(models)))
        for i in range(len(models)):
            for j in range(len(models)):
                win[i, j] = np.nan if i == j else (a[i] > a[j]).mean()
        cmap = DIV.copy()
        cmap.set_bad(SURFACE)
        im = ax.imshow(np.ma.masked_where(np.isnan(win), win), cmap=cmap, vmin=0, vmax=1)
        for i in range(len(models)):
            for j in range(len(models)):
                if i == j:
                    continue
                v = win[i, j]
                ax.text(j, i, f"{100 * v:.0f}", ha="center", va="center", fontsize=7.8,
                        color="white" if abs(v - 0.5) > 0.22 else INK)
        ax.set_xticks(range(len(models)), [LABEL[m] for m in models], rotation=30, ha="right", fontsize=8)
        ax.set_yticks(range(len(models)), [LABEL[m] for m in models], fontsize=8)
        ax.set_title(f"x{scale}", loc="left")
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(length=0)
    fig.tight_layout()
    save(fig, "sr_fig10_winmatrix")


fig_winmatrix()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 11 -- quality against parameter count
# ---------------------------------------------------------------------------
# Parameters read off the saved checkpoints (not re-typed from the papers), against test
# PSNR. The horizontal position is the model, the vertical span is what scale buys from the
# same weights: SwinIR at x8 costs the same as SwinIR at x2 and is 8 dB worse, which is the
# trade a deployment actually makes.

In [ ]:
def param_count(model, scale):
    import torch
    ckpt = torch.load(RUNS / run_name(model, scale) / "weights" / "best.pt", map_location="cpu")
    return sum(p.numel() for p in ckpt["model"].values()) / 1e6


def fig_cost():
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    for model in NAMES:
        pts = [(param_count(model, s), TEST[(model, s)]["psnr"], s)
               for s in SCALES if (model, s) in TEST]
        if not pts:
            continue
        xs, ys, ss = zip(*pts)
        ax.plot(xs, ys, color=COLOR[model], lw=1.0, alpha=0.5, zorder=1)
        for x, y, s in pts:
            ax.plot(x, y, "o", ms=6, color=COLOR[model], mec="white", mew=1.2, zorder=3)
            ax.annotate(f"x{s}", (x, y), xytext=(5, -9), textcoords="offset points",
                        fontsize=7, color=MUTED)

    ax.set_xscale("log")
    # The legend sits above the panel: inside it, the SwinIR column at the right edge and its
    # x2/x4/x8 labels are what it covers.
    fig.legend(handles=[Line2D([], [], color=COLOR[m], marker="o", lw=1.0, label=LABEL[m])
                        for m in NAMES if any((m, s) in TEST for s in SCALES)],
               loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0),
               handlelength=1.2, columnspacing=1.0, fontsize=8)
    ax.set_xlabel("parameters (millions, log scale)")
    ax.set_ylabel("test PSNR (dB)")
    x0, x1 = ax.get_xlim()
    ax.set_xlim(x0 / 1.35, x1 * 1.30)  # room for the scale labels of the end columns
    tidy(ax)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save(fig, "sr_fig11_cost")


fig_cost()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 12 -- the results table as a heat map
# ---------------------------------------------------------------------------
# The same three numbers as figure 5, arranged the way the table is (models x scales) and
# shaded, so a reader sees the pattern before reading a digit: PSNR and SSIM darken towards
# x2 and towards the better model; MSE does the opposite, and is shaded separately for that
# reason.

In [ ]:
def fig_table():
    metrics = [("psnr", "PSNR (dB)", "PSNR (dB)", False),
               ("ssim", "SSIM", "SSIM", False),
               ("mse", "MSE", "MSE (×10⁻³)", True)]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3))
    for n, (ax, (key, label, title, invert)) in enumerate(zip(axes, metrics)):
        mat = np.array([[TEST.get((m, s), {}).get(key, np.nan) for s in SCALES] for m in NAMES])
        shown = -mat if invert else mat
        im = ax.imshow(shown, cmap=SEQ, aspect="auto")
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if np.isnan(mat[i, j]):
                    continue
                v = mat[i, j]
                # MSE is shown in units of 1e-3: five decimals do not fit a cell, and the
                # panel title says which unit is being read.
                fmt = f"{v:.2f}" if key == "psnr" else (f"{v:.3f}" if key == "ssim" else f"{v * 1000:.2f}")
                rng = np.nanmax(shown) - np.nanmin(shown)
                dark = (shown[i, j] - np.nanmin(shown)) > 0.55 * rng
                ax.text(j, i, fmt, ha="center", va="center", fontsize=7.4,
                        color="white" if dark else INK)
        ax.set_xticks(range(len(SCALES)), [f"x{s}" for s in SCALES])
        # The row names appear once, on the left panel: repeating them three times leaves
        # the cells too narrow to read.
        ax.set_yticks(range(len(NAMES)))
        ax.set_yticklabels([LABEL[m] for m in NAMES] if n == 0 else [])
        ax.set_title(title, loc="left")
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(length=0)
    fig.tight_layout()
    save(fig, "sr_fig12_table")


fig_table()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 13 -- where the detail comes from: radial power spectra
# ---------------------------------------------------------------------------
# The average azimuthal power spectrum of the luma channel over the rendered crops, one
# panel per scale, in dB relative to the ground truth's low frequencies. A model that only
# smooths (bicubic, and any model behaving like it) falls away from the target at high
# frequency; a model that invents texture (SRGAN at x8) can sit *above* the target out
# there, which is hallucinated detail and is why its PSNR does not follow. The x2 curves
# hug the target to near Nyquist; the x8 curves separate early and stay separated.

In [ ]:
def radial_power(image, bins=48):
    """Azimuthally averaged power spectrum of the luma channel, and the bin frequencies."""
    y = (0.299 * image[..., 0] + 0.587 * image[..., 1] + 0.114 * image[..., 2]) / 255.0
    win = np.hanning(y.shape[0])[:, None] * np.hanning(y.shape[1])[None, :]
    spec = np.abs(np.fft.fftshift(np.fft.fft2((y - y.mean()) * win))) ** 2
    n = y.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    radius = np.hypot(yy - n // 2, xx - n // 2) / n  # cycles per pixel, 0 .. 0.5
    which = np.clip((radius * 2 * bins).astype(int), 0, bins - 1)
    count = np.bincount(which.ravel(), minlength=bins)
    mean = np.bincount(which.ravel(), weights=spec.ravel(), minlength=bins) / np.maximum(count, 1)
    return mean, (np.arange(bins) + 0.5) / (2 * bins)


def fig_spectrum():
    keys_by_scale = {s: [k for k in MANIFEST["keys"] if f"x{s}" in MANIFEST["boxes"][k]]
                     for s in SCALES}
    fig, axes = plt.subplots(1, len(SCALES), figsize=(7.2, 2.9), sharey=True)
    for ax, scale in zip(axes, SCALES):
        keys = keys_by_scale[scale]
        if not keys:
            ax.set_visible(False)
            continue
        sources = [("hr", "Ground truth", INK)] + [("bicubic", "Bicubic baseline", MUTED)] + \
                  [(m, LABEL[m], COLOR[m]) for m in NAMES
                   if all(has_crop(m, k, scale) for k in keys)]
        curves = {}
        freq = None
        for source, _, _ in sources:
            acc = []
            for key in keys:
                img = (truth(key, scale) if source == "hr" else
                       bicubic(key, scale) if source == "bicubic" else crop(source, key, scale))
                p, freq = radial_power(img)
                acc.append(np.log10(np.maximum(p, 1e-12)))
            curves[source] = np.mean(acc, axis=0)
        ref = np.mean(curves["hr"][:3])
        for source, label, color in sources:
            ax.plot(freq, 10 * (curves[source] - ref), color=color, lw=1.8 if source in ("hr", "bicubic") else 1.5,
                    ls="-" if source == "hr" else ((0, (3, 2)) if source == "bicubic" else "-"),
                    label=label, zorder=4 if source == "hr" else 3)
        ax.axvline(0.5 / scale, color=AXIS, lw=0.8)
        ax.text(0.5 / scale, ax.get_ylim()[0] + 1, f"LR Nyquist (x{scale})", fontsize=6.2,
                color=MUTED, rotation=90, va="bottom", ha="right")
        ax.set_xlim(0, 0.5)
        ax.set_xlabel("frequency (cycles / pixel)", fontsize=8.5)
        ax.set_title(f"x{scale}", loc="left")
        tidy(ax)
    axes[0].set_ylabel("power (dB, relative to target)", fontsize=9)
    handles = [Line2D([], [], color=INK, lw=1.8, label="Ground truth"),
               Line2D([], [], color=MUTED, lw=1.5, ls=(0, (3, 2)), label="Bicubic baseline")] + \
              [Line2D([], [], color=COLOR[m], lw=1.5, label=LABEL[m]) for m in NAMES
               if all(all(has_crop(m, k, sc) for k in keys_by_scale[sc])
                      for sc in SCALES if keys_by_scale[sc])]
    fig.legend(handles=handles, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.0),
               handlelength=1.6, columnspacing=1.1, fontsize=8)
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    save(fig, "sr_fig13_spectrum")


if MANIFEST:
    fig_spectrum()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 14 -- halfway between the output and the truth
# ---------------------------------------------------------------------------
# One x8 crop split down the middle: the model on the left, the ground truth on the right.
# The seam is where the eye compares, and at x8 it shows exactly what the network is
# making up -- a roofline that stays crisp across the seam is real, one that changes
# texture across it is not.

In [ ]:
def fig_split(scale=8):
    keys = [k for k in MANIFEST["keys"] if f"x{scale}" in MANIFEST["boxes"][k]]
    if not keys:
        return
    key = max(keys, key=lambda k: MANIFEST["boxes"][k][f"x{scale}"]["psnr"].get(f"swinir_x{scale}", 0))
    psnr = MANIFEST["boxes"][key][f"x{scale}"]["psnr"]
    models = [m for m in NAMES if f"{m}_x{scale}" in psnr]
    # Two panes per row: at four across, a 512 px crop is 1.8 in wide and the seam -- the
    # only thing this figure is about -- stops being visible.
    ncol = 2
    nrow = (len(models) + ncol - 1) // ncol
    side = 7.2 / ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(7.2, side * nrow + 0.35), squeeze=False)
    target = truth(key, scale)
    for idx, model in enumerate(models):
        ax = axes[idx // ncol][idx % ncol]
        out = crop(model, key, scale)
        n = out.shape[1]
        canvas = np.concatenate([out[:, : n // 2], target[:, n // 2:]], axis=1)
        strip(ax, canvas)
        ax.plot([n / 2 - 0.5] * 2, [0, n - 1], color="white", lw=1.0, zorder=5)
        ax.text(0.25, -0.02, LABEL[model], transform=ax.transAxes, ha="center", va="top",
                fontsize=9, color=INK)
        ax.text(0.75, -0.02, "ground truth", transform=ax.transAxes, ha="center", va="top",
                fontsize=9, color=MUTED)
    for idx in range(len(models), nrow * ncol):
        axes[idx // ncol][idx % ncol].axis("off")
    fig.subplots_adjust(left=0.01, right=0.99,
                        top=1 - 0.30 / fig.get_figheight(), bottom=0.30 / fig.get_figheight(),
                        wspace=0.03, hspace=0.10)
    save(fig, f"sr_fig14_split_x{scale}")


if MANIFEST:
    fig_split()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 15 -- the same detail at x2, x4 and x8 on three kinds of content
# ---------------------------------------------------------------------------
# The split pane of figure 14, one scene per row and one scale per column, on a 256 px
# window cut out of the rendered crop, so the differences survive print size: at x2 a model
# is essentially the target, and at x8 it is drawing plausible texture on top of a 256 px
# input. The three scenes are not hand-picked -- they are the crops with the most vegetation,
# the most asphalt and the most concrete in the rendered pool, which is the closest thing to
# "a field, a road and a roof" that statistics over the pixels can give. The pixel statistics
# are printed, so a scene can be swapped for a better one by hand and the reason recorded.
# In every pane the left half is the model and the right half is the ground truth; the badge
# names the model (the best one at that scale) and gives the PSNR of the 256 px window, not
# of the whole crop. Figure 16 is the same grid with all four models side by side.

In [ ]:
# `content_scenes` / `_content_features` live with figure 3 -- the same three scenes.


def _best_model(key, scale):
    """The model with the highest crop PSNR at that scale."""
    psnr = MANIFEST["boxes"][key][f"x{scale}"]["psnr"]
    return max((m for m in NAMES if f"{m}_x{scale}" in psnr), key=lambda m: psnr[f"{m}_x{scale}"])


def _best_window(hr, size):
    """Top-left of the `size` window with the most gradient energy, on a 4x4 grid."""
    g = hr.astype(np.float32).mean(-1)
    gx, gy = np.gradient(g)
    energy = np.abs(gx) + np.abs(gy)
    step = (hr.shape[0] - size) // 4
    best, best_xy = -1.0, (0, 0)
    for y in range(5):
        for x in range(5):
            e = energy[y * step:y * step + size, x * step:x * step + size].mean()
            if e > best:
                best, best_xy = e, (y * step, x * step)
    return best_xy


def _window_psnr(out, hr, win):
    a = out[win].astype(np.float32) / 255.0
    b = hr[win].astype(np.float32) / 255.0
    d = float(np.mean((a - b) ** 2))
    return 99.0 if d == 0 else 10 * np.log10(1.0 / d)


def _split_pane(ax, key, model, scale, win, badge_name=True, size=256):
    """Left half the model, right half the ground truth, seam marked."""
    if not has_crop(model, key, scale):
        ax.axis("off")
        return
    hr, out = truth(key, scale), crop(model, key, scale)
    strip(ax, np.concatenate([out[win][:, : size // 2], hr[win][:, size // 2:]], axis=1))
    ax.plot([size / 2 - 0.5] * 2, [0, size - 1], color="white", lw=1.0, zorder=5)
    if badge_name:
        badge(ax, LABEL[model], "upper left", size=6.6)
    badge(ax, f"{_window_psnr(out, hr, win):.1f} dB", "lower right", size=6.6)


def _windows(scenes, size=256):
    """One window per scene, taken from the x8 crop: the rendered box is the same physical
    region at every scale, so the window has to be chosen once and reused."""
    return {key: _best_window(truth(key, 8), size) for key in scenes.values()}


def _sub(key, win, size=256):
    return (slice(win[0], win[0] + size), slice(win[1], win[1] + size))


def fig_zoom(size=256):
    scenes = content_scenes()
    if not scenes:
        return
    wins = _windows(scenes, size)
    fig, axes = plt.subplots(len(scenes), len(SCALES), squeeze=False,
                             figsize=(7.2, 7.2 / len(SCALES) * len(scenes) + 0.45))
    for i, (cls, key) in enumerate(scenes.items()):
        for j, scale in enumerate(SCALES):
            ax = axes[i, j]
            _split_pane(ax, key, _best_model(key, scale), scale, _sub(key, wins[key]), size=size)
            if i == 0:
                ax.set_title(f"x{scale}", loc="center", fontsize=10)
            if j == 0:
                ax.set_ylabel(cls, fontsize=9, color=INK, labelpad=6)
    fig.subplots_adjust(left=0.075, right=0.995, top=1 - 0.30 / fig.get_figheight(),
                        bottom=0.02, wspace=0.03, hspace=0.05)
    save(fig, "sr_fig15_zoom")


if MANIFEST:
    fig_zoom()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 16 -- all four models at one scale, on the same three windows
# ---------------------------------------------------------------------------
# Figure 15 with the column meaning changed from scale to model, and one file per scale:
# rows are still the three kinds of content, columns are the four models, and every pane is
# still split between the model and the target. This is the figure that answers "which one
# should I use", where 15 answers "what does the scale cost". The window per scene is the
# same one figure 15 uses, so the two can be read side by side.

In [ ]:
def fig_zoom_models(scale, size=256):
    scenes = content_scenes() if not hasattr(fig_zoom_models, "_scenes") else fig_zoom_models._scenes
    fig_zoom_models._scenes = scenes
    if not scenes:
        return
    wins = _windows(scenes, size)
    have = [m for m in NAMES if any(has_crop(m, k, scale) for k in scenes.values())]
    fig, axes = plt.subplots(len(scenes), len(have), squeeze=False,
                             figsize=(7.2, 7.2 / len(have) * len(scenes) + 0.45))
    for i, (cls, key) in enumerate(scenes.items()):
        for j, model in enumerate(have):
            ax = axes[i, j]
            _split_pane(ax, key, model, scale, _sub(key, wins[key]), badge_name=False, size=size)
            if i == 0:
                ax.set_title(LABEL[model], loc="center", fontsize=9.5)
            if j == 0:
                ax.set_ylabel(cls, fontsize=9, color=INK, labelpad=6)
    fig.subplots_adjust(left=0.075, right=0.995, top=1 - 0.30 / fig.get_figheight(),
                        bottom=0.02, wspace=0.03, hspace=0.05)
    save(fig, f"sr_fig16_models_x{scale}")


if MANIFEST:
    for _s in SCALES:
        fig_zoom_models(_s)

In [ ]:
# ---------------------------------------------------------------------------
# Figure 17 -- the paper's training figure: what was selected, and how it got there
# ---------------------------------------------------------------------------
# Two panels, because the paper gives this figure one column width and the two statements
# belong to one claim: the validation loss each checkpoint was *selected* on, and the curve it
# was selected from. It replaces the earlier `training_curves.pdf` (loss/PSNR/SSIM against
# epoch in three line panels, which repeated the results table). A middle panel of test
# metrics was tried and dropped: every number in it is in the results table, one page away.
#
# (a) is a three-dimensional bar chart, which is normally a way to make a reader compare
# heights through a perspective. Two things stop it from being one here: the projection is
# orthographic (equal height is equal length anywhere in the plot), and every bar carries its
# own value, so nothing is read off a slope. What the panel is for is the *scale* effect, a
# 3.5x difference between x2 and x8 that no small misreading of a height can hide.
#
# (b) is the convergence that (a) cannot show, in a single pair of axes: twelve runs, colour
# for the model, line style for the scale, and the dot on each curve the epoch `best.pt` was
# written from -- the checkpoint panel (a) describes. SRGAN at x4 is why the dot matters: it
# selects epoch 15 of 60 and is worse by the last one, and no "final epoch" summary says so.
# Twelve series in one axes were nearly twelve small multiples instead; see `_panel_convergence`
# for what made the single axes work.
#
# Both panels are coloured by *model* only (the palette of the rest of the notebook,
# validated for CVD); scale is position and line style, never a second colour scale.
#
# Every size here is written for the *printed* figure. The paper sets this at
# `\includegraphics[width=\linewidth]`, and its \linewidth is 372 pt = 5.17 in, so a figure
# drawn on 7.2 in would print at 0.72x and a 6 pt label would land at 4.3 pt. That is why the
# fonts below are ~1.4x what the rest of the notebook uses, and why the labels are abbreviated:
# the type is correct on the page, and the space for it comes out of the text, not the data.

#: Abbreviated names, for this figure only. "Autoencoder" at print size is 40 % wider than the
#: strip these panels leave for a row label and pushed into the panel beside it; the caption
#: spells out what AE is.
SHORT = {"autoencoder": "AE", "srcnn": "SRCNN", "srgan": "SRGAN", "swinir": "SwinIR"}

In [ ]:
def _selected_row(model, scale):
    """The `metrics.csv` row `best.pt` was written from: the lowest finite validation loss.
    Returns (row, all rows), so a panel can use either the selection or the curve."""
    rows = CURVE.get((model, scale))
    if not rows:
        return None, None
    ok = [r for r in rows if np.isfinite(r["val_loss"])]
    return (min(ok, key=lambda r: r["val_loss"]) if ok else None), rows


def _panel_loss_surface(ax):
    """Validation loss at the selected checkpoint, on the (model x scale) grid."""
    ax.set_proj_type("ortho")          # equal heights render equal; no perspective rescaling
    ax.view_init(elev=30, azim=-62)
    # mplot3d's own depth sorting paints these bars in the order they were added, so the back
    # row lands on top of the front row and buries its labels. Sorted by hand instead: with
    # the camera at azim -62 the near corner is large x, small y, so that is the last drawn.
    ax.computed_zorder = False
    cells = []
    for mi, model in enumerate(NAMES):
        for si, scale in enumerate(SCALES):
            row, _ = _selected_row(model, scale)
            if row is not None:
                cells.append((0.47 * mi - 0.88 * si, mi, si, model, row["val_loss"] * 1e3))
    cells.sort(reverse=True)
    zmax = max(c[4] for c in cells) if cells else 1.0
    for _, mi, si, model, h in cells:
        ax.bar3d(mi + 0.36, si + 0.36, 0, 0.28, 0.28, h, color=COLOR[model],
                 edgecolor=SURFACE, linewidth=0.5, shade=True, alpha=0.96, zorder=2)
    for _, mi, si, model, h in cells:
        # Haloed: in this projection a bar can stand behind the label of the bar below it,
        # and ink on a saturated bar is unreadable. The white pad keeps the number on top.
        ax.text(mi + 0.50, si + 0.50, h + zmax * 0.095, f"{h:.1f}",
                ha="center", va="bottom", fontsize=7.6, color=INK2, zorder=20,
                bbox=dict(boxstyle="round,pad=0.10", fc=(1, 1, 1, 0.80), ec="none"))
    ax.set_xticks(np.arange(len(NAMES)) + 0.50, [SHORT[m] for m in NAMES])
    ax.set_yticks(np.arange(len(SCALES)) + 0.50, [f"$\\times${s}" for s in SCALES])
    ax.set_xticklabels([SHORT[m] for m in NAMES], fontsize=8.4, rotation=-52,
                       ha="right", rotation_mode="anchor")
    ax.set_xlim(0, len(NAMES)); ax.set_ylim(0, len(SCALES))
    # zoom: a 3-D axes reserves room for the tick labels of two axes plus the box corners, and
    # at 1.6 in wide the data ended up in the lower half of its slot. The zoom fills it.
    ax.set_box_aspect((1.46, 0.88, 1.22), zoom=1.46)
    ax.set_zlim(0, zmax * 1.22)
    ax.set_zticks([0, 3, 6, 9, 12])
    ax.tick_params(labelsize=8.4, pad=0, length=0)
    ax.tick_params(axis="x", pad=6)
    ax.tick_params(axis="y", pad=2)
    # No z-axis caption: "x10-3" is in the panel title, which keeps this edge free for the
    # numbers alone -- the panel is 2 inches wide and a rotated caption here costs a bar.
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor(SURFACE)   # no box: the data is the only ink with weight
        axis._axinfo["grid"].update(color=HAIR, linewidth=0.5)


def _panel_convergence(ax):
    """Validation PSNR against epoch for all twelve runs, in one pair of axes.

    Twelve series would normally be twelve small multiples -- and those were tried here, as a
    4 x 3 grid of sparklines, which was readable but read as twelve figures. In one pair of
    axes the three scales separate by themselves -- they sit ~2.5 dB apart -- so the panel
    reads as three bands with no border between them. Colour carries the model (the family
    colour used everywhere in this notebook) and the line style carries the scale -- solid x2,
    dashed x4, dotted x8 -- with the three band labels at the right edge doubling as the style
    key. SRGAN is the one series that leaves its band: its adversarial objective swings far
    enough that x2 falls into the x4 band and x8 climbs into it, which is a fact about that run
    and not a reading error, so it is drawn as it is rather than clipped back into place."""
    styles = {2: "-", 4: (0, (4.2, 1.5)), 8: (0, (1.1, 1.3))}
    bands = {}
    for mi, model in enumerate(NAMES):
        for scale in SCALES:
            sel, rows = _selected_row(model, scale)
            if not rows:
                continue
            ep = np.array([r["epoch"] for r in rows])
            val = np.array([r["val_psnr"] for r in rows])
            # Smooth only, on a 5-epoch window: at twelve series the raw trace of SRGAN alone
            # is enough ink to hide the other eleven, and the epoch-to-epoch noise is not what
            # the panel is for. Five epochs still leaves SRGAN's x4 peak at epoch 16 visible.
            ax.plot(ep, _smooth(val, 5), color=COLOR[model], ls=styles[scale], lw=1.5, zorder=3)
            if sel is not None:
                ax.plot([sel["epoch"]], [sel["val_psnr"]], "o", ms=4.2, color=COLOR[model],
                        mec="white", mew=0.8, zorder=4)
            band = bands.setdefault(scale, [])
            band.extend(r["val_psnr"] for r in rows if np.isfinite(r["val_psnr"]))
    # Cut at 40 epochs, the budget every run shares: SRGAN trains for 60, and the extra 20
    # epochs are the subject of one sentence in the paper, not of the panel.
    ax.set_xlim(0, 40)
    ax.set_xticks([0, 10, 20, 30, 40])
    ax.set_xlabel("epoch", fontsize=8.6, color=INK2, labelpad=2)
    ax.set_ylabel("validation PSNR (dB)", fontsize=8.6, color=INK2, labelpad=2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(False)
    ax.tick_params(labelsize=7.8, length=2, pad=1, colors=MUTED)
    # The scale key lives at the right margin: a short sample of the line style the scale uses,
    # then the factor. Both are in the panel's ink rather than the muted grey of a tick label --
    # at the print size these three are the only thing naming the scales, so they have to read
    # as labels and not as debris.
    for scale, lo, hi in ((s_, min(bands[s_]), max(bands[s_]))
                          for s_ in SCALES if s_ in bands):
        # x in axes coordinates, y in data coordinates: the label has to sit at the band's own
        # height in dB, which transAxes would misread as a fraction.
        ty = ax.get_yaxis_transform()
        y = (lo + hi) / 2
        ax.plot([1.012, 1.062], [y, y], transform=ty, color=INK2, ls=styles[scale], lw=1.5,
                clip_on=False, solid_capstyle="butt")
        ax.text(1.078, y, f"$\\times${scale}", transform=ty, ha="left", va="center",
                fontsize=10.0, color=INK)


def _legend_models(ax):
    """The four model colours, on one row above the panel; the styles are keyed at the right."""
    handles = [Line2D([], [], color=COLOR[m], lw=2.0, label=SHORT[m]) for m in NAMES]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.005), ncol=4,
              frameon=False, fontsize=8.6, handlelength=1.5, columnspacing=1.6,
              handletextpad=0.5, borderpad=0.0)


def fig_paper_curves():
    fig = plt.figure(figsize=(7.2, 2.75))
    outer = fig.add_gridspec(1, 2, width_ratios=[0.94, 1.06], wspace=0.34,
                             left=0.030, right=0.902, bottom=0.145, top=0.845)
    ax_a = fig.add_subplot(outer[0], projection="3d")
    # zoomed, the 3-D box overflows its slot to the right and the z tick labels land on panel
    # (b)'s y label; nudging the axes back by the same amount keeps them in the gap.
    ax_a.set_position(ax_a.get_position().translated(-0.012, 0))
    ax_b = fig.add_subplot(outer[1])
    _panel_loss_surface(ax_a)
    _panel_convergence(ax_b)
    _legend_models(ax_b)
    # A 3-D axes does not place a title the way a 2-D one does, so the panel labels are figure
    # text at one height, anchored on each panel's left edge.
    for x0, text in ((ax_a.get_position().x0, "(a) selected validation loss, $\\times10^{-3}$"),
                     (ax_b.get_position().x0, "(b) validation PSNR over training")):
        fig.text(x0, 0.965, text, fontsize=10.6, color=INK, fontweight="bold",
                 ha="left", va="center")
    save(fig, "sr_fig17_paper_training")


fig_paper_curves()

## Choosing between them

The paper needs three of these, not fourteen. A combination that argues the story without
repeating itself:

- **One results figure.** 5a (dots) carries PSNR, SSIM and MSE with the per-image spread, at
  the size of a single-column table and without a zero baseline. 12 is the same data with
  the digits, if the table should stay a table; 5b only if the venue expects bars.
- **One qualitative figure.** 2 (four scenes at x4, six columns) is the workhorse. 3 (one
  scene down the scales) is the better figure if the paper's claim is about the *decay* with
  scale rather than about the models; 15 is the same claim compressed to one column width --
  three kinds of content down, the three scales across, each pane split between the model
  and the target; 16 is the model comparison at one scale, in the same three windows, one
  file per scale (use the x8 one if only one fits); 4 is the one to use if a reviewer asks
  where the error is; 14 belongs on the project page, not in the paper.
- **One analysis figure.** 13 (power spectra) is the strongest statement this task can make
  that a table cannot: it shows bicubic and the L2 models collapsing at high frequency and
  SRGAN sitting above the target, which is exactly why its PSNR does not win. If the paper
  argues about robustness rather than detail, 7 (CDF: the models fail on different images)
  or 8 (how often a model is worse than bicubic) is the substitute.

9 is supplementary material, 10 is for a discussion of per-image variance, 11 is for the
deployment-cost sentence, and 1 is the same message as 5a in the smallest possible space --
useful as a talk slide.

**17 is the one the paper currently uses**, included as`sr_fig17_paper_training.pdf`: the loss each run wasselected on, and the validation curve it was selected from -- the loss half of 9 next to all
of 9, in one figure. 5, 9 and 12 stay here as the reference versions a reviewer may ask for.